In [1]:
import pyreadr
import pandas as pd
import numpy as np
import math
from collections import defaultdict, Counter
import os
from utils import plot_tree as pt
import webbrowser


# Calcolo dei pesi dei geni

La metrica per calcolare i pesi è calcolata assegnando a ciascun gene un punteggio. Ogni fattore contribuisce attraverso un peso specifico e il punteggio finale del gene è ottenuto come prodotto di tali pesi.

Il punteggio del singolo gene è definito come:


$$
Weight_{mutation} =
w_{status}
\times
b_{PCAWG}
\times
k_{severity}
\times
wgd_{factor}
$$



dove:

- $w_{status}$: peso associato alla tipologia di mutazione
- $b_{PCAWG}$: indica se il gene è riconosciuto come driver nel catalogo PCAWG
- $k_{severity}$: rappresenta la severità del cariotipo
- $wgd_{factor}$: corregge il punteggio in base alla presenza di un evento di Whole Genome Doubling (WGD).

Il peso finale del gene viene ottenuto sommando i contributi di tutte le sue mutazioni:

$$
Weight_{gene} =
\sum_{i=1}^{m}
Weight_{mutation_i}
$$

ovvero:

$$
Weight_{gene} =
\sum_{i=1}^{m}
\left(
w_{status,i}
\times
b_{PCAWG,i}
\times
k_{severity,i}
\times
wgd_{factor,i}
\right)
$$

dove $m$ rappresenta il numero di mutazioni osservate per quel gene all'interno dello stesso **clock rank**.

In [2]:
# Caricamento dei dati dal file RDS
print("Caricamento dataset in corso...")
result = pyreadr.read_r('06_Cb_BTM_table.rds')
df = result[None]

# Filtraggio
ttype_selezionato = "OV"  #selezione del tumore
df_filtrato = df[df["ttype"] == ttype_selezionato].copy()

# Eliminazione geni sani (WT) e dati mancanti critici
df_filtrato = df_filtrato[df_filtrato['mutatation_status'] != "WT"]
df_filtrato = df_filtrato.dropna(subset=["clock_rank", "gene"])

print(f"Dataset filtrato per {ttype_selezionato}. Righe totali da analizzare: {len(df_filtrato)}")

Caricamento dataset in corso...
Dataset filtrato per OV. Righe totali da analizzare: 54


In [3]:
#analisi Nan trovati
righe_scartate = df_filtrato[df_filtrato[["clock_rank", "gene"]].isna().any(axis=1)]
totale_scartate = len(righe_scartate)

print(f"--- ANALISI DROPNA ---")
print(f"Righe totali che verranno rimosse: {totale_scartate}")

if totale_scartate > 0:
    # analisi dei Nan per tipologia
    solo_clock_na = (righe_scartate['clock_rank'].isna()) & (righe_scartate['gene'].notna())
    solo_gene_na = (righe_scartate['gene'].isna()) & (righe_scartate['clock_rank'].notna())
    entrambi_na = (righe_scartate['clock_rank'].isna()) & (righe_scartate['gene'].isna())

    print("\nDettaglio colonne mancanti:")
    print(f"- Manca solo 'clock_rank' (il gene c'è ma non è stato datato): {solo_clock_na.sum()}")
    print(f"- Manca solo 'gene' (datato ma senza nome gene): {solo_gene_na.sum()}")
    print(f"- Mancano entrambi: {entrambi_na.sum()}")

    # Analisi dei Nan per tipologia di mutazione
    print("\nStatus mutazionale ('mutatation_status') delle righe rimosse:")
    print(righe_scartate['mutatation_status'].value_counts().to_string())
    print("-" * 22 + "\n")


# return df filtrato
df_filtrato = df_filtrato.dropna(subset=["clock_rank", "gene"])

print(f"Dataset filtrato per {ttype_selezionato}. Righe totali valide da analizzare: {len(df_filtrato)}")

--- ANALISI DROPNA ---
Righe totali che verranno rimosse: 0
Dataset filtrato per OV. Righe totali valide da analizzare: 54


In [4]:
# funzioni per la mappatura dei pesi

# Mappatura tipologia mutazione per distaccare i driver principali
status_weights = {'CI_M': 1.0, 'M': 2.0, 'CNA_driver': 4.0}
df_filtrato['w_status'] = df_filtrato['mutatation_status'].map(status_weights).fillna(1.0)

# Moltiplicatore PCAWG (2.0 se il catalogo conferma il driver, 1.0 altrimenti)
df_filtrato['b_pcawg'] = np.where(df_filtrato['mutation_call'].notna(), 2.0, 1.0)

# Moltiplicatore Cariotipo (LOH o instabilità numerica)
def analizza_cariotipo(k_str):
    if pd.isna(k_str): 
        return 1.0
    try:
        parts = str(k_str).split(':')
        major, minor = int(parts[0]), int(parts[1])
        if minor == 0: 
            return 1.5  # Incremento per Loss of Heterozygosity
        if (major + minor) >= 5: 
            return 1.5  # Incremento per amplificazione massiccia
    except:
        pass
    return 1.0

df_filtrato['k_severity'] = df_filtrato['karyotype'].apply(analizza_cariotipo)

# Fattore WGD globale del paziente
df_filtrato['wgd_factor'] = np.where(df_filtrato['is_WGD'] == True, 0.8, 1.0)

# Calcolo del peso finale della singola riga (istanza mutazionale)
df_filtrato['gene_weight'] = (df_filtrato['w_status'] * df_filtrato['b_pcawg'] * df_filtrato['k_severity'] * df_filtrato['wgd_factor'])

Spiegazione calcolo dei pesi:
1. tipologia di mutazione `w_status`: scala esponenziale (non lineare), perchè le mutazioni driver sono più gravi rispetto a quelle puntiformi:
    - `CI_M` (*1.0*): mutazione con meno importanza, non si è certi della natura clonale (Clonal Illusion), resta uguale  
    - `M` (*2.0*): mutazione puntiforme certa (cambio del nucleotide=lettera) 
    - `CNA_driver` (*4.0*): mutazione più grave, variazione del numero di copie di un gene driver (amplificazione/delezione)

2. Gene driver `b_pcawg`: gene riconosciuto come driver dal catalogo ufficiale internazionale (PCAWG):
    - driver già riconosciuto (*2*): se si sa già che quel gene è driver e cioè causa il tumore, la probabilità che stia guidando l'evoluzione di questo tumore è alta
    - driver non riconosciuto (*1*): invariabile

3. Karyotype `k_severity`: legge la stringa del cariotipo (2:1, etc) e separa il numero di copie dell'allele maggiore e dell'allele minore - pesi:
    - minor=0 (LOH_ Loss of Heterozygosity): perdita di una delle due copie alleliche, è un evento catastrofico  = *1.5*; 
    - major+minor>=5:  elevata amplificazione del locus = *1.5*; 
    - cariotipo trisomia (2:1) o bilanciato (2:2): = *1*.

4. raddoppiamento globale `wgd_factor`: controlla se l'intero paziente ha subito un whole genome doubling (raddoppiamento dei 46 cromosomi)- pesi:
    - tumori WGD (*0.8*): la presenza di alterazioni del numero di copie è più frequente, il peso viene attenuato
    - tumori classici (is_WGD==F): genoma normale, resta invariata (*1*)

Quindi se ho due geni diversi nello stesso clock rank:


- Gene A (Non Driver): Status M (*2.0*) * Non in PCAWG (*1.0*) * Cariotipo 2:1 (*1.0*) * WGD Presente (*0.8*) = Punteggio 1.6

- Gene B (Driver): Status CNA_driver (*4.0*) * In PCAWG (*2.0*) * LOH Presente (*1.5*) * No WGD (*1.0*) = Punteggio 12.0


In [5]:
df_filtrato = df_filtrato.sort_values(by=['clock_rank', 'gene', 'w_status'], ascending=[True, True, False])

# Aggregation
# Somma automaticamente le mutazioni multiple dello stesso gene nello stesso rank e tra diversi pazienti
node_weights = df_filtrato.groupby(['clock_rank', 'gene']).agg(
    gene_weight=('gene_weight', 'sum'),                  # Somma i pesi per il punteggio totale
    mut_status=('mutatation_status', 'first'),           # Conserva la tipologia più grave (grazie all'ordinamento di prima)
    n_pazienti=('sample_id', 'nunique'),                 # Conta in quanti pazienti DIVERSI compare
    n_occorrenze=('gene', 'count')                       # Conta il numero totale di mutazioni subite in assoluto
).reset_index()
# df con gene_weight in ordine decrescente
node_weights.sort_values(by='gene_weight', ascending=False).head(10)

,clock_rank,gene,gene_weight,mut_status,n_pazienti,n_occorrenze
27,2,TP53,18.0,M,3,3
15,1,RB1,18.0,CNA_driver,3,3
11,1,NF1,12.0,CNA_driver,2,2
3,1,BRCA1,12.0,CNA_driver,2,2
0,1,ARID1A,12.0,CNA_driver,2,2
17,1,STAG1,9.0,CI_M,5,9
6,1,CDK12,8.0,M,1,2
8,1,KIF2A,6.5,CI_M,4,5
12,1,NF2,6.0,M,1,1
20,2,ARID1B,6.0,M,1,1


Spiegazione raggruppamento: 
- `df_filtrato.groupby(['clock_rank', 'gene'])`: raggruppa i dati in base a quando è avvenuta la mutaizone (clock_rank) e il nome del gene (gene)
- `['gene_weight'].sum()`: per ogni gene e il rank corrispondente (TP53 al rank1), somma tutti i pesi presenti, importante pk:
se un paziente ha più mutazioni puntiformi sullo stesso gene, avvenute nello stesso clock_rank, ci saranno le stesse corrispondenti righe nel df dei pesi. Questa funzione sommerà i pesi di queste righe, considerando un valore complessivo per quel gene in quel paziente (e per tutti i pazienti che hanno quel gene in quel rank)
- `reset_index()`: riporta indici a colonne normali, otteniamo un df (node_weights) con 3 colonne: clock_rank, gene e somma dei pesi (`gene_weight`)

- poi il df viene ordinato, basandosi sulla colonna `gene_weight`, in ordine decrescente (punteggio più alto, al più basso)

# Costruzione dell’albero


L’albero viene costruito organizzando i geni all’interno di ciascun `clock_rank` e selezionando i nodi in base al loro `gene_weight`:

- Nodi: geni mutati, ordinati in modo decrescente per peso - in cima dovrebbero esserci i CNA_driver condivisi da molti pazienti, in fondo le M più rare; numeri di geni selezionati nello stesso clock: 
    - numero >20: ne vengono selezionati solo alcuni (radice quadrata) - es. se ce ne sono 100, prende i top 10
    - sennò i primi 15

- Archi: i geni appartenenti a rank differenti vengono collegati seguendo la struttura evolutiva
 



In [6]:
# Selezione e Filtraggio dei nodi per l'albero
multi_tree = defaultdict(list)
ranks = node_weights['clock_rank'].unique()

for r in sorted(ranks):
    # Ordinamento dei geni decrescenti per il rank corrente
    df_rank = node_weights[node_weights['clock_rank'] == r].sort_values(by='gene_weight', ascending=False)
    genes_list = list(zip(df_rank['gene'], df_rank['gene_weight'], df_rank['mut_status'], df_rank['n_pazienti']))
    
    # Applicazione del taglio matematico (Top 5 o radice quadrata)
    if len(genes_list) > 20:
        genes_list = genes_list[0:int(math.sqrt(len(genes_list)))]
    else:
        genes_list = genes_list[0:15]
    
    # Salvataggio nel dizionario con etichetta formattata per Graphviz
    for gene, weight, status, pts in genes_list:
        label = f"{gene}\n[{status}]\nW= {weight:.1f}, #= {pts}" #nome e peso
        multi_tree[r].append(label)

print("Struttura dell'albero multi-paziente completata con successo.")

Struttura dell'albero multi-paziente completata con successo.


Spiegazione: 
1. per ogni cluster temporale (clock), i geni appartenenti ad esso vengono isolati e ordinati dal più grave al meno grave (gene_weight decrescente), in cima dovrebbero esserci i CNA_driver condivisi da molti pazienti, in fondo le M più rare

2. se ci sono più di 20 geni mutati nello stesso clock, ne vengono selezionati solo alcuni (radice quadrata) per la grafica dell'albero. (es. se ce ne sono 100, prende i top 10). Sennò i primi 15.

In [7]:
try:
    primo_rank = min(multi_tree.keys())
    print(f"\n--- ELENCO GENI INIZIALI (RANK {primo_rank}) ---")
    for nodo in multi_tree[primo_rank]:
        testo_pulito = nodo.replace('\n', ' ')
        print(f"- {testo_pulito}")
    print("---------------------------------------\n")
except ValueError:
    print("Attenzione: nessun nodo trovato nel rank iniziale.")


--- ELENCO GENI INIZIALI (RANK 1) ---
- RB1 [CNA_driver] W= 18.0, #= 3
- BRCA1 [CNA_driver] W= 12.0, #= 2
- ARID1A [CNA_driver] W= 12.0, #= 2
- NF1 [CNA_driver] W= 12.0, #= 2
- STAG1 [CI_M] W= 9.0, #= 5
- CDK12 [M] W= 8.0, #= 1
- KIF2A [CI_M] W= 6.5, #= 4
- TP53 [M] W= 6.0, #= 1
- NF2 [M] W= 6.0, #= 1
- LATS1 [M] W= 6.0, #= 1
- CBFB [M] W= 6.0, #= 1
- CREBBP [CNA_driver] W= 6.0, #= 1
- KMT2D [M] W= 4.0, #= 1
- PIK3CA [M] W= 4.0, #= 1
- ATM [CI_M] W= 2.0, #= 2
---------------------------------------



In [8]:
# Generazione del Grafo Visivo

file_name = f"multi_patient_weighted_{ttype_selezionato}"
# funzione gagiornata
pt.print_tree(multi_tree, file_name)


print("\nProcedura completata! Controlla la cartella 'produced'.")


Procedura completata! Controlla la cartella 'produced'.


In [9]:
multi_tree = defaultdict(list)
ranks = sorted(node_weights['clock_rank'].unique())

for r in ranks:
    df_rank = node_weights[node_weights['clock_rank'] == r].sort_values(by='gene_weight', ascending=False)
    genes_list = list(zip(df_rank['gene'], df_rank['gene_weight'], df_rank['mut_status'], df_rank['n_pazienti']))
    
    if len(genes_list) > 20:
        genes_list = genes_list[0:int(math.sqrt(len(genes_list)))]
    else:
        genes_list = genes_list[0:15]
    
    for gene, weight, status, pts in genes_list:
        #filter the tree with genes which appear in more than 1 patient
        if r > 2 or pts > 1:
            label = f"{gene}\n[{status}]\nW= {weight:.1f} | Pts= {pts}"
            multi_tree[r].append(label)

print("Struttura dell'albero filtrato completata!")

Struttura dell'albero filtrato completata!


In [10]:
# ==========================================
file_name = f"multi_patient_weighted_filtered_{ttype_selezionato}"

# La tua chiamata semplice e diretta:
pt.print_tree(multi_tree, file_name)

# Domande: 
1. tutti i geni mutati sono collegati con tutti. Se il gene è presente solo in un paziente, come facciamo ad essere sicuri che appartenga all'ordine evolutivo del tumore? o sia solo "casuale"? 

Possibile soluzione:

Creiamo un altro albero in cui nel rank 1 e nel Rank 2 seleziona solo i geni mutati in più di 1 paziente. Dal Rank3 in poi, non c'è differenza. 

2. Se lo stesso gene mutato risulta in più clock_rank, come facciamo a dire a quale appartiene?
Possibile soluzione
In base al numero di pazienti in cui risulta mutato e al suo peso 